<a href="https://colab.research.google.com/github/cactus1386/NationalCard-ImageProccessing/blob/main/trainOCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install hezar

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from hezar.models import Model
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from PIL import Image
import os
import yaml

In [4]:
class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super(CNNBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding)
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = nn.functional.relu(x)
        return x

In [9]:
class CRNN(nn.Module):
    def __init__(self, n_channels, num_classes, map2seq_in_dim, map2seq_out_dim, rnn_dim):
        super(CRNN, self).__init__()
        self.cnn = nn.ModuleDict({
            '0': CNNBlock(n_channels, 64),
            '2': CNNBlock(64, 128),
            '4': CNNBlock(128, 256),
            '5': CNNBlock(256, 256),
            '7': CNNBlock(256, 512),
            '8': CNNBlock(512, 512),
            '10': CNNBlock(512, 512, kernel_size=3, padding=1)  # تغییر به kernel_size=3
        })
        self.map2seq = nn.Linear(map2seq_in_dim, map2seq_out_dim)
        self.rnn1 = nn.LSTM(map2seq_out_dim, rnn_dim, bidirectional=True, batch_first=True)
        self.rnn2 = nn.LSTM(rnn_dim * 2, rnn_dim, bidirectional=True, batch_first=True)
        self.classifier = nn.Linear(rnn_dim * 2, num_classes)

    def forward(self, x):
        for key in sorted(self.cnn.keys(), key=lambda k: int(k)):
            x = self.cnn[key](x)
            if key in ['0', '2']:
                x = nn.functional.max_pool2d(x, 2)  # (2, 2) pooling
            elif key in ['5', '8']:
                x = nn.functional.max_pool2d(x, (2, 1))  # (2, 1) pooling
        batch, channels, height, width = x.size()
        x = x.permute(0, 3, 1, 2).reshape(batch, width, -1)
        x = self.map2seq(x)
        x, _ = self.rnn1(x)
        x, _ = self.rnn2(x)
        x = self.classifier(x)
        return x

In [6]:
def preprocess_image(image_path):
    with open("/content/drive/MyDrive/crnn-fa-printed-96-long/preprocessor/image_processor_config.yaml", "r") as f:
        config = yaml.safe_load(f)

    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((32, 384)),
        transforms.Lambda(lambda x: x.transpose(Image.FLIP_LEFT_RIGHT) if config["mirror"] else x),
        transforms.ToTensor(),
        transforms.Normalize(mean=config["mean"], std=config["std"]),
        transforms.Lambda(lambda x: x * config["rescale"])
    ])
    image = Image.open(image_path).convert("L")
    return transform(image)

In [7]:
class NewOCRDataset(Dataset):
    def __init__(self, custom_data):
        self.custom_data = custom_data
        self.image_paths = list(custom_data.keys())

        with open("/content/drive/MyDrive/crnn-fa-printed-96-long/model_config.yaml", "r") as f:
            config = yaml.safe_load(f)
        self.char2id = {char: idx for idx, char in config["id2label"].items()}

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = preprocess_image(img_path)
        label = self.custom_data[img_path]
        label_ids = [self.char2id.get(c, 0) for c in label]
        return image, torch.tensor(label_ids, dtype=torch.long), len(label_ids)

In [16]:
def collate_fn(batch):
    images, targets, target_lengths = zip(*batch)
    images = torch.stack(images, dim=0)  # تصاویر را به یک تنسور واحد تبدیل می‌کند
    targets = [t.clone().detach() for t in targets]  # تارگت‌ها به صورت لیست باقی می‌مانند
    target_lengths = torch.tensor(target_lengths, dtype=torch.long)
    return images, targets, target_lengths

In [17]:
def train_model(model, dataloader, num_epochs=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    criterion = nn.CTCLoss(blank=0, zero_infinity=True)
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for batch_idx, (images, targets, target_lengths) in enumerate(dataloader):
            images = images.to(device)
            # تارگت‌ها را به یک تنسور واحد تبدیل می‌کنیم
            targets = torch.cat(targets).to(device)
            target_lengths = target_lengths.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            outputs = outputs.log_softmax(2)  # برای CTC Loss باید خروجی‌ها log softmax شوند

            # CTC Loss انتظار دارد که خروجی‌ها به شکل (T, N, C) باشند
            # خروجی مدل ما (N, T, C) است، پس باید آن را permute کنیم
            outputs = outputs.permute(1, 0, 2)

            # طول ورودی‌ها (input_lengths) برابر با طول دنباله خروجی مدل است
            input_lengths = torch.full((images.size(0),), outputs.size(0), dtype=torch.long).to(device)
            loss = criterion(outputs, targets, input_lengths, target_lengths)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx}], Loss: {loss.item():.4f}")

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch [{epoch+1}/{num_epochs}] completed, Avg Loss: {avg_loss:.4f}")

    torch.save(model.state_dict(), "finetuned.pt")
    print("مدل ذخیره شد: finetuned.pt")

In [18]:
with open("/content/drive/MyDrive/crnn-fa-printed-96-long/model_config.yaml", "r") as f:
        config = yaml.safe_load(f)

model = CRNN(
    n_channels=config["n_channels"],
    map2seq_in_dim=config["map2seq_in_dim"],
    map2seq_out_dim=config["map2seq_out_dim"],
    rnn_dim=config["rnn_dim"],
    num_classes=len(config["id2label"])
)

model.load_state_dict(torch.load("/content/drive/MyDrive/crnn-fa-printed-96-long/model.pt"))
model.eval()

custom_data = {
    "/content/drive/MyDrive/images/ocrImage/code.png": "۹۴۹۰۵۴۹۰۰۸",
    "/content/drive/MyDrive/images/ocrImage/birth.png": "۱۳۶۷/۱۱/۲۳",
    "/content/drive/MyDrive/images/ocrImage/date.png": "۱۴۱۰/۰۳/۲۰",
    '/content/drive/MyDrive/images/ocrImage/father.png': 'اکبر',
    '/content/drive/MyDrive/images/ocrImage/last.png' : 'کرمی',
    "/content/drive/MyDrive/images/ocrImage/name.png": "تیرداد",
}

dataset = NewOCRDataset(custom_data)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
train_model(model, dataloader, num_epochs=5)

Epoch [1/5], Batch [0], Loss: 3.4429
Epoch [1/5], Batch [1], Loss: 9.1975
Epoch [1/5], Batch [2], Loss: 5.9214
Epoch [1/5] completed, Avg Loss: 6.1873
Epoch [2/5], Batch [0], Loss: 5.5010
Epoch [2/5], Batch [1], Loss: 3.3543
Epoch [2/5], Batch [2], Loss: 3.9602
Epoch [2/5] completed, Avg Loss: 4.2718
Epoch [3/5], Batch [0], Loss: 1.2508
Epoch [3/5], Batch [1], Loss: 2.4401
Epoch [3/5], Batch [2], Loss: 2.9159
Epoch [3/5] completed, Avg Loss: 2.2023
Epoch [4/5], Batch [0], Loss: 1.3802
Epoch [4/5], Batch [1], Loss: 0.5463
Epoch [4/5], Batch [2], Loss: 1.3471
Epoch [4/5] completed, Avg Loss: 1.0912
Epoch [5/5], Batch [0], Loss: 0.4633
Epoch [5/5], Batch [1], Loss: 0.8007
Epoch [5/5], Batch [2], Loss: 0.8831
Epoch [5/5] completed, Avg Loss: 0.7157
مدل ذخیره شد: finetuned.pt
